# Joint Supervised-SSL Model Experiment
### for Assignment 2: SSL
**By Tahiyat Chy**

This model combines the main MAE model with the Supervised baseline to test whether it can achieve higher accuracy than

## Initial Setup

In [1]:
# Core imports
from pathlib import Path
import random
import time
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu130
CUDA available: True


In [2]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

Mounted at /content/gdrive


In [3]:
# Path set to assignment directory

PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169/assignment2')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

Project root: /content/gdrive/MyDrive/ITCS_6169/assignment2
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/test


In [4]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Data Preparation
Define data augmentations. Split the training images into training and validation set.

In [5]:
import torchvision.transforms as T

BATCH_SIZE = 64
IMG_SIZE = 224
NUM_CLASSES = 16

# Data augmentation
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandAugment(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_train_dataset = datasets.ImageFolder(root=TRAIN_ROOT)

# Split into train and val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# Implement custom wrapper to apply different transforms to train and val subsets
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = TransformSubset(train_subset, transform=train_transforms)
val_dataset = TransformSubset(val_subset, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")

Total training images: 1920
Total validation images: 480


## Model Architecture


In [6]:
class ConvStemPatchEmbed(nn.Module):
    """Hierarchical 3-layer Convolutional Stem for small datasets."""
    def __init__(self, in_channels=3, embed_dim=192):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 48, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.GELU(),
            nn.Conv2d(48, 96, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.GELU(),
            nn.Conv2d(96, embed_dim, kernel_size=3, stride=4, padding=1)
        )

    def forward(self, x):
        x = self.stem(x) # (B, embed_dim, 14, 14)
        return x.flatten(2).transpose(1, 2) # (B, 196, embed_dim)


class JointViTMAE(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=16,
        embed_dim=192,
        depth=6,
        num_heads=6,             # Increased head count from 4 to 6 for finer attention splits
        decoder_embed_dim=128,
        decoder_depth=2,
        decoder_num_heads=4
    ):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = in_channels * patch_size * patch_size

        # 1. Multi-Stage ConvStem
        self.patch_embed = ConvStemPatchEmbed(in_channels=in_channels, embed_dim=embed_dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))

        # 2. Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            batch_first=True, activation='gelu', dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # 3. Head
        self.head = nn.Linear(embed_dim, num_classes)

        # 4. MAE Decoder
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim))

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_embed_dim, nhead=decoder_num_heads, dim_feedforward=decoder_embed_dim * 4,
            batch_first=True, activation='gelu'
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=decoder_depth)
        self.decoder_pred = nn.Linear(decoder_embed_dim, self.patch_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)

    def patchify_and_normalize(self, x):
        """Converts raw images into per-patch zero-mean, unit-variance target vectors."""
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.view(B, C, H // p, p, W // p, p)
        patches = patches.permute(0, 2, 4, 3, 5, 1).contiguous()
        patches = patches.view(B, self.num_patches, self.patch_dim)

        # Per-patch Target Normalization
        mean = patches.mean(dim=-1, keepdim=True)
        var = patches.var(dim=-1, keepdim=True)
        patches_norm = (patches - mean) / (torch.sqrt(var) + 1e-6)
        return patches_norm

    def forward_encoder(self, x, mask_ratio):
        B = x.shape[0]
        patches_target = self.patchify_and_normalize(x)

        x_embed = self.patch_embed(x) + self.pos_embed[:, 1:, :]

        if mask_ratio > 0:
            len_keep = int(self.num_patches * (1 - mask_ratio))
            noise = torch.rand(B, self.num_patches, device=x.device)
            ids_shuffle = torch.argsort(noise, dim=1)
            ids_restore = torch.argsort(ids_shuffle, dim=1)
            ids_keep = ids_shuffle[:, :len_keep]

            x_masked = torch.gather(x_embed, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, x_embed.shape[-1]))
        else:
            x_masked = x_embed
            ids_restore = None

        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        x_enc_in = torch.cat((cls_token.expand(B, -1, -1), x_masked), dim=1)
        x_enc_out = self.encoder(x_enc_in)
        return x_enc_out, patches_target, ids_restore

    def forward_decoder(self, x_enc_out, ids_restore):
        B = x_enc_out.shape[0]
        cls_tok = x_enc_out[:, :1, :]
        latent = x_enc_out[:, 1:, :]

        latent_dec = self.decoder_embed(latent)
        len_keep = latent_dec.shape[1]

        mask_tokens = self.mask_token.repeat(B, self.num_patches - len_keep, 1)
        x_full = torch.cat([latent_dec, mask_tokens], dim=1)
        x_full = torch.gather(x_full, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x_full.shape[-1]))

        x_full = x_full + self.decoder_pos_embed[:, 1:, :]
        cls_dec = self.decoder_embed(cls_tok) + self.decoder_pos_embed[:, :1, :]

        x_dec_out = self.decoder(torch.cat([cls_dec, x_full], dim=1))
        return self.decoder_pred(x_dec_out[:, 1:, :])

    def forward(self, x, mask_ratio=0.75):
        if self.training and mask_ratio > 0:
            x_enc_out, patches_target, ids_restore = self.forward_encoder(x, mask_ratio)
            cls_logits = self.head(x_enc_out[:, 0])
            pred_patches = self.forward_decoder(x_enc_out, ids_restore)
            return cls_logits, pred_patches, patches_target, ids_restore
        else:
            x_enc_out, _, _ = self.forward_encoder(x, mask_ratio=0.0)
            return self.head(x_enc_out[:, 0])

In [7]:
def compute_mae_loss(pred_patches, target_patches, ids_restore, mask_ratio=0.75):
    """Computes MSE reconstruction loss strictly across masked patches."""
    B, N, _ = pred_patches.shape
    len_keep = int(N * (1 - mask_ratio))

    # Calculate MSE per patch
    loss_patch = ((pred_patches - target_patches) ** 2).mean(dim=-1)

    # Generate binary mask matrix (0 = kept, 1 = masked)
    mask = torch.ones([B, N], device=pred_patches.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)

    # Mean loss calculated strictly on masked patches
    return (loss_patch * mask).sum() / mask.sum()

## Training Loop

In [8]:
def train_joint_model(model, train_loader, val_loader, epochs=100, lr=5e-4, lambda_recon=0.5, mask_ratio=0.75):
    cls_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)

    # 5-epoch Linear Warmup + Cosine Annealing
    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 0.5 * (1.0 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss, running_cls_loss, running_recon_loss = 0.0, 0.0, 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            cls_logits, pred_patches, target_patches, ids_restore = model(inputs, mask_ratio=mask_ratio)

            loss_cls = cls_criterion(cls_logits, labels)
            loss_recon = compute_mae_loss(pred_patches, target_patches, ids_restore, mask_ratio)

            # Weight reconstruction loss (lambda=0.5 balances gradients smoothly)
            loss = loss_cls + lambda_recon * loss_recon

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_cls_loss += loss_cls.item() * inputs.size(0)
            running_recon_loss += loss_recon.item() * inputs.size(0)

        scheduler.step()

        total_samples = len(train_loader.dataset)
        epoch_loss = running_loss / total_samples
        epoch_cls = running_cls_loss / total_samples
        epoch_recon = running_recon_loss / total_samples

        model.eval()
        corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)

        epoch_acc = corrects.double() / len(val_loader.dataset)

        if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch+1:03d}/{epochs} | Total Loss: {epoch_loss:.4f} (Cls: {epoch_cls:.4f}, Recon: {epoch_recon:.4f}) | Val Acc: {epoch_acc:.4f}")

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    print(f"\nTraining complete. Best Validation Accuracy: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

In [9]:
# Instantiate model
model = JointViTMAE(num_classes=NUM_CLASSES).to(device)
print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

joint_model = train_joint_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    lr=5e-4,
    lambda_recon=0.5, # Reduced recon factor balances feature representation
    mask_ratio=0.75
)

Total Model Parameters: 3,465,088
Epoch 005/100 | Total Loss: 2.5328 (Cls: 2.1305, Recon: 0.8046) | Val Acc: 0.3875
Epoch 010/100 | Total Loss: 2.4067 (Cls: 2.0104, Recon: 0.7926) | Val Acc: 0.5125
Epoch 015/100 | Total Loss: 2.1539 (Cls: 1.7618, Recon: 0.7842) | Val Acc: 0.5687
Epoch 020/100 | Total Loss: 2.0441 (Cls: 1.6555, Recon: 0.7773) | Val Acc: 0.6396
Epoch 025/100 | Total Loss: 1.9531 (Cls: 1.5634, Recon: 0.7794) | Val Acc: 0.5938
Epoch 030/100 | Total Loss: 1.8820 (Cls: 1.4931, Recon: 0.7778) | Val Acc: 0.6542
Epoch 035/100 | Total Loss: 1.8272 (Cls: 1.4401, Recon: 0.7741) | Val Acc: 0.7042
Epoch 040/100 | Total Loss: 1.7831 (Cls: 1.3969, Recon: 0.7724) | Val Acc: 0.6521
Epoch 045/100 | Total Loss: 1.6734 (Cls: 1.2863, Recon: 0.7742) | Val Acc: 0.6875
Epoch 050/100 | Total Loss: 1.6243 (Cls: 1.2384, Recon: 0.7717) | Val Acc: 0.7333
Epoch 055/100 | Total Loss: 1.6122 (Cls: 1.2281, Recon: 0.7682) | Val Acc: 0.7125
Epoch 060/100 | Total Loss: 1.5045 (Cls: 1.1201, Recon: 0.7687) 

In [10]:
"""
EXPERIMENT: Fine-Tuning the Joint Model on Unmasked Patched using Classification Loss
"""
def finetune_joint_model(model, train_loader, val_loader, epochs=30, lr=1e-4):
    cls_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    # Lower learning rate (1e-4) to preserve representations learned in Phase 1
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            # Set mask_ratio=0.0 to train on 100% visible patches without reconstruction
            cls_logits = model(inputs, mask_ratio=0.0)
            loss = cls_criterion(cls_logits, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        scheduler.step()

        total_samples = len(train_loader.dataset)
        epoch_loss = running_loss / total_samples

        # Validation Step
        model.eval()
        corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs, mask_ratio=0.0)
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)

        epoch_acc = corrects.double() / len(val_loader.dataset)

        print(f"Fine-Tune Epoch {epoch+1:02d}/{epochs} | Cls Loss: {epoch_loss:.4f} | Val Acc: {epoch_acc:.4f}")

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    print(f"\nFine-Tuning complete. Best Validation Accuracy: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

# Execute Stage fine-Tuning on the best joint model weights
finetuned_model = finetune_joint_model(
    model=joint_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,
    lr=1e-4
)

Fine-Tune Epoch 01/30 | Cls Loss: 0.8659 | Val Acc: 0.7958
Fine-Tune Epoch 02/30 | Cls Loss: 0.8333 | Val Acc: 0.7937
Fine-Tune Epoch 03/30 | Cls Loss: 0.8281 | Val Acc: 0.7729
Fine-Tune Epoch 04/30 | Cls Loss: 0.8219 | Val Acc: 0.7771
Fine-Tune Epoch 05/30 | Cls Loss: 0.8060 | Val Acc: 0.7771
Fine-Tune Epoch 06/30 | Cls Loss: 0.7887 | Val Acc: 0.7937
Fine-Tune Epoch 07/30 | Cls Loss: 0.7836 | Val Acc: 0.7917
Fine-Tune Epoch 08/30 | Cls Loss: 0.7861 | Val Acc: 0.7833
Fine-Tune Epoch 09/30 | Cls Loss: 0.7989 | Val Acc: 0.7771
Fine-Tune Epoch 10/30 | Cls Loss: 0.7810 | Val Acc: 0.7958
Fine-Tune Epoch 11/30 | Cls Loss: 0.7829 | Val Acc: 0.7750
Fine-Tune Epoch 12/30 | Cls Loss: 0.7858 | Val Acc: 0.7833
Fine-Tune Epoch 13/30 | Cls Loss: 0.7408 | Val Acc: 0.7896
Fine-Tune Epoch 14/30 | Cls Loss: 0.7635 | Val Acc: 0.7875
Fine-Tune Epoch 15/30 | Cls Loss: 0.7582 | Val Acc: 0.7875
Fine-Tune Epoch 16/30 | Cls Loss: 0.7414 | Val Acc: 0.7979
Fine-Tune Epoch 17/30 | Cls Loss: 0.7288 | Val Acc: 0.78